# Estimação com vários snapshots (IEEE 13 barras, trecho RG60–632)

Usa a base gerada em `Gera_Snapshots.ipynb`: 10 000 vetores θ (espalhamento de 1 %), cada um com K = 5 snapshots de carga diferente. Os casos comparados são:

* **1 snapshot**: só o snapshot 0 de cada θ, como no dataset original;
* **5 snapshots**: as entradas dos 5 snapshots concatenadas (5 × 36 = 180 colunas).

Em cada caso, rodamos sem ruído e com o ruído nominal do artigo (α = 1). A µPMU é sorteada de novo a cada snapshot. Os TIs têm erro sistemático fixo por canal, igual em todos os snapshots, porque os instrumentos são os mesmos.

As referências são:
* **média do treino**: devolver sempre a média dos θ de treino, ou seja, não estimar nada;
* **Bayes pela equação da linha**: média a posteriori com os snapshots empilhados, e o MAPE mínimo teórico (aproximação gaussiana).

A divisão 80/20 é feita por θ, então os snapshots de um mesmo θ ficam todos no treino ou todos no teste.

In [1]:
import time, sys, pickle, gc
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import gerador_snapshots as g
sys.path.insert(0, str(Path('..', '..', 'DadosBelvedere').resolve()))
import feederlib as fl

SEED, ALPHA_NOM = 0, 1.0
ARQ = Path('snapshots_0.01_K5.csv.gz')
CACHE = Path('resultados_snapshots.pkl')
RECALCULAR = False

## 1. Dados e entradas

In [2]:
df = pd.read_csv(ARQ).sort_values(['theta_id', 'snapshot'])
K = df.snapshot.nunique(); n = df.theta_id.nunique()
Y = df.groupby('theta_id')[g.PARAMS].first().values

def fasores(pre, sufs):
    return np.stack([fl._fasor(df[f'{pre}{s[0]}'], df[f'{pre}{s[1]}']) for s in sufs], 1).reshape(n, K, 3)
D = dict(V1=fasores('V', [(f'{j}_mag_RG60', f'{j}_ang_RG60') for j in (1, 2, 3)]),
         V2=fasores('V', [(f'{j}_mag_632', f'{j}_ang_632') for j in (1, 2, 3)]),
         I1=fasores('I1', [(f'_mag_fase{p}', f'_ang_fase{p}') for p in 'ABC']),
         I2=fasores('I2', [(f'_mag_fase{p}', f'_ang_fase{p}') for p in 'ABC']))

import opendssdirect as dss
dss.Text.Command(f'redirect [{g.MASTER}]'); dss.Lines.Name(g.LINHA)
Y_MEIO = 1j * 2 * np.pi * 60 * np.array(dss.Lines.CMatrix()).reshape(3, 3) * dss.Lines.Length() * 1e-9 / 2
ELL = g.COMPRIMENTO_FT / 5280
idx_tr, idx_te = train_test_split(np.arange(n), test_size=0.2, random_state=SEED)
print(f'{n} θ × {K} snapshots; treino {len(idx_tr)}, teste {len(idx_te)}')

10000 θ × 5 snapshots; treino 8000, teste 2000


In [3]:
def com_ruido(D, alpha, it_seed, e_pmu):
    """Aplica o ruído amostra a amostra (n·K linhas); TIs fixos por canal em todos os snapshots."""
    plano = {c: D[c].reshape(-1, 3) for c in fl.CANAIS}
    out = fl.aplicar_ruido(plano, e_pmu, fl.erros_ti(it_seed), alpha) if alpha > 0 else plano
    return {c: out[c].reshape(n, K, 3) for c in fl.CANAIS}

def entradas(F, snaps):
    """|V|, ∠V, |I|, ∠I, P, Q nos dois terminais (config 'both'), concatenados por snapshot."""
    Fp = {c: F[c][:, snaps].reshape(-1, 3) for c in fl.CANAIS}
    return fl.montar_X(Fp, 'both').reshape(n, -1)

def bayes(F, snaps, alpha, idx):
    """Média a posteriori com os snapshots empilhados e MAPE mínimo teórico."""
    mu, Sig = Y[idx_tr].mean(0), np.diag(Y[idx_tr].var(0))
    med, sd = np.empty((len(idx), 12)), np.empty((len(idx), 12))
    Is = F['I1'] - F['V1'] @ Y_MEIO.T
    for m, i in enumerate(idx):
        J = np.vstack([fl.jacobiano(Is[i, k], ELL) for k in snaps])
        dV = F['V1'][i, snaps] - F['V2'][i, snaps]
        y = np.concatenate([np.r_[v.real, v.imag] for v in dV])
        R = np.zeros((6 * len(snaps),) * 2)
        if alpha > 0:
            Rk = fl.cov_ruido_dV(D['V1'][i, snaps], D['V2'][i, snaps], Is[i, snaps], alpha, 1)
            for j in range(len(snaps)):
                R[6 * j:6 * j + 6, 6 * j:6 * j + 6] = Rk[j]
        S = J @ Sig @ J.T + R + 1e-12 * np.eye(len(R))
        G = Sig @ J.T @ np.linalg.inv(S)
        med[m] = mu + G @ (y - J @ mu)
        sd[m] = np.sqrt(np.clip(np.diag(Sig - G @ J @ Sig), 0, None))
    return med, 100 * np.sqrt(2 / np.pi) * np.mean(sd / np.abs(Y[idx]), axis=0)

## 2. Experimentos

In [4]:
CASOS = {'1 snapshot': [0], f'{K} snapshots': list(range(K))}

def rodar(nome_caso, snaps, alpha):
    e_pmu = fl.erros_pmu(n * K, 1, np.random.default_rng(SEED))
    F = com_ruido(D, alpha, 1, e_pmu)
    X = entradas(F, snaps)
    keep = X[idx_tr].std(0) > 0
    sx, sy = StandardScaler().fit(X[idx_tr][:, keep]), StandardScaler().fit(Y[idx_tr])
    Xtr, Xte = sx.transform(X[idx_tr][:, keep]), sx.transform(X[idx_te][:, keep])
    Yte = Y[idx_te]
    out = {'Média do treino (sem estimação)': fl.mape(Yte, np.broadcast_to(Y[idx_tr].mean(0), Yte.shape))}
    for nome, m in fl.novos_modelos().items():
        t0 = time.time()
        m.fit(Xtr, sy.transform(Y[idx_tr]))
        out[nome] = fl.mape(Yte, sy.inverse_transform(m.predict(Xte)))
        print(f'  {nome_caso}, α={alpha}: {nome} {out[nome].mean():.4f} %  ({time.time()-t0:.0f} s)', flush=True)
        del m; gc.collect()
    med, lim = bayes(F, snaps, alpha, idx_te)
    out['Bayes (equação da linha)'] = fl.mape(Yte, med)
    out['Limite teórico (equação da linha)'] = lim
    return pd.DataFrame(out, index=g.PARAMS)

if CACHE.exists() and not RECALCULAR:
    RES = pickle.load(open(CACHE, 'rb'))
else:
    RES = {}
    for alpha in (0.0, ALPHA_NOM):
        for caso, snaps in CASOS.items():
            RES[(caso, alpha)] = rodar(caso, snaps, alpha)
    pickle.dump(RES, open(CACHE, 'wb'))

## 3. Resultados

MAPE médio dos 12 parâmetros (%) no conjunto de teste. A linha **Média do treino** é o que se obtém sem estimar nada: um método só é útil se ficar abaixo dela.

In [5]:
ORDEM = ['Média do treino (sem estimação)', 'Ridge', 'MLP', 'LightGBM', 'RandomForest',
         'Bayes (equação da linha)', 'Limite teórico (equação da linha)']
tab = {}
for (caso, alpha), t in RES.items():
    rotulo = 'sem ruído' if alpha == 0 else 'com ruído'
    tab[(rotulo, caso)] = t.mean().reindex(ORDEM)
T = pd.DataFrame(tab).reindex(ORDEM)
T.columns = pd.MultiIndex.from_tuples(T.columns)
T.round(4)

sem ruído              com ruído  \
                                  1 snapshot 5 snapshots 1 snapshot   
Média do treino (sem estimação)       0.7979      0.7979     0.7979   
Ridge                                 0.5340      0.5337     0.7525   
MLP                                   0.5393      0.5638     0.7609   
LightGBM                              0.7054      0.7082     0.7743   
RandomForest                          0.7375      0.7632     0.7795   
Bayes (equação da linha)              0.5300      0.0004     0.7976   
Limite teórico (equação da linha)     0.5297      0.0048     0.7980   

                                               
                                  5 snapshots  
Média do treino (sem estimação)        0.7979  
Ridge                                  0.6747  
MLP                                    0.7147  
LightGBM                               0.7387  
RandomForest                           0.7745  
Bayes (equação da linha)               0.8340  
Limite teórico (equação da linha)      0.7933

In [6]:
# ganho em relação à média do treino: MAPE do método / MAPE da média (abaixo de 1 = a estimação ajuda)
(T / T.loc[ORDEM[0]]).round(3)

sem ruído              com ruído  \
                                  1 snapshot 5 snapshots 1 snapshot   
Média do treino (sem estimação)        1.000       1.000      1.000   
Ridge                                  0.669       0.669      0.943   
MLP                                    0.676       0.707      0.954   
LightGBM                               0.884       0.888      0.970   
RandomForest                           0.924       0.957      0.977   
Bayes (equação da linha)               0.664       0.000      1.000   
Limite teórico (equação da linha)      0.664       0.006      1.000   

                                               
                                  5 snapshots  
Média do treino (sem estimação)         1.000  
Ridge                                   0.846  
MLP                                     0.896  
LightGBM                                0.926  
RandomForest                            0.971  
Bayes (equação da linha)                1.045  
Limite teórico (equação da linha)       0.994

MAPE por grupo de parâmetros (resistências próprias e mútuas, reatâncias próprias e mútuas) com 5 snapshots:

In [7]:
grupos = {'r próprias': ['r11', 'r22', 'r33'], 'r mútuas': ['r12', 'r13', 'r23'],
          'x próprias': ['x11', 'x22', 'x33'], 'x mútuas': ['x12', 'x13', 'x23']}
pg = {}
for alpha in (0.0, ALPHA_NOM):
    t = RES[(f'{K} snapshots', alpha)]
    for gr, ps in grupos.items():
        pg[('sem ruído' if alpha == 0 else 'com ruído', gr)] = t.loc[ps].mean()[[c for c in ORDEM if c in t]]
pd.DataFrame(pg).round(4)

sem ruído                               \
                                  r próprias r mútuas x próprias x mútuas   
Média do treino (sem estimação)       0.8084   0.7898     0.7974   0.7961   
Ridge                                 0.6186   0.7646     0.2370   0.5146   
MLP                                   0.6507   0.7788     0.2861   0.5396   
LightGBM                              0.7930   0.7931     0.5459   0.7007   
RandomForest                          0.8035   0.7930     0.7070   0.7494   
Bayes (equação da linha)              0.0002   0.0002     0.0006   0.0005   
Limite teórico (equação da linha)     0.0031   0.0019     0.0088   0.0052   

                                   com ruído                               
                                  r próprias r mútuas x próprias x mútuas  
Média do treino (sem estimação)       0.8084   0.7898     0.7974   0.7961  
Ridge                                 0.7669   0.7859     0.4772   0.6690  
MLP                                   0.7978   0.8060     0.5475   0.7075  
LightGBM                              0.8039   0.7972     0.6193   0.7344  
RandomForest                          0.8107   0.7932     0.7295   0.7646  
Bayes (equação da linha)              0.8196   0.7989     0.8474   0.8701  
Limite teórico (equação da linha)     0.7926   0.8019     0.7834   0.7951

## 4. Vários snapshots com informação física

Na seção anterior, os modelos recebiam só |V|, ∠V, |I|, ∠I, P e Q, e não aproveitavam os snapshots extras. Obter θ exige resolver o sistema $\Delta V^{(k)} = \ell Z I^{(k)}$, cujos coeficientes são as próprias medições, e isso não é uma função linear das entradas. Aqui entregamos ao modelo o que a física já calcula a partir dos 5 snapshots:

* **ΔV e corrente série** por snapshot (partes real e imaginária): $\Delta V = V_1 - V_2$ e $I_s = I_1 - (Y/2)V_1$;
* **θ por mínimos quadrados (MQ)**: a solução do sistema empilhado (30 equações, 12 incógnitas);
* **θ por Bayes**: a média a posteriori da equação da linha, que puxa o MQ para o valor a priori quando o ruído domina.

Os conjuntos de entradas comparados são *física* (só os itens acima) e *original + física*. Treino e teste usam o mesmo conjunto de TIs (semente 1), como na seção 3. A RandomForest fica de fora por causa do tempo (10 min por ajuste com 5 snapshots).

In [8]:
CACHE_FIS = Path('resultados_snapshots_fisica.pkl')
SNAPS = list(range(K))

def fisicas(F, alpha, idx):
    """ΔV e I_s por snapshot, θ por MQ e θ por Bayes (tudo a partir das medições, com ou sem ruído)."""
    V1, V2, I1 = (F[c][idx] for c in ('V1', 'V2', 'I1'))
    m = len(idx)
    Is = I1 - V1 @ Y_MEIO.T
    dV = V1 - V2
    J = np.stack([fl.jacobiano(i, ELL) for i in Is.reshape(-1, 3)]).reshape(m, K * 6, 12)
    y = np.concatenate([dV.real, dV.imag], axis=2).reshape(m, K * 6)
    JT = np.swapaxes(J, 1, 2)
    th_mq = np.linalg.solve(JT @ J, (JT @ y[..., None]))[..., 0]
    mu, Sig = Y[idx_tr].mean(0), np.diag(Y[idx_tr].var(0))
    R = np.zeros((m, 6 * K, 6 * K))
    if alpha > 0:
        Rk = fl.cov_ruido_dV(V1.reshape(-1, 3), V2.reshape(-1, 3), Is.reshape(-1, 3), alpha, 1).reshape(m, K, 6, 6)
        for k in range(K):
            R[:, 6 * k:6 * k + 6, 6 * k:6 * k + 6] = Rk[:, k]
    S = J @ Sig @ JT + R + 1e-12 * np.eye(6 * K)
    th_bayes = mu + (Sig @ JT @ np.linalg.solve(S, (y - J @ mu)[..., None]))[..., 0]
    base = np.hstack([dV.real.reshape(m, -1), dV.imag.reshape(m, -1), Is.real.reshape(m, -1), Is.imag.reshape(m, -1)])
    return np.hstack([base, th_mq, th_bayes]), th_mq, th_bayes

def rodar_fisica(alpha):
    e_pmu = fl.erros_pmu(n * K, 1, np.random.default_rng(SEED))
    F = com_ruido(D, alpha, 1, e_pmu)
    Xf, th_mq, th_b = fisicas(F, alpha, np.arange(n))
    Xo = entradas(F, SNAPS)
    Yte = Y[idx_te]
    out = {'Média do treino (sem estimação)': fl.mape(Yte, np.broadcast_to(Y[idx_tr].mean(0), Yte.shape)),
           'θ por MQ (5 snapshots)': fl.mape(Yte, th_mq[idx_te]),
           'θ por Bayes (5 snapshots)': fl.mape(Yte, th_b[idx_te])}
    conjuntos = {'física': Xf, 'original + física': np.hstack([Xo, Xf])}
    for cj, X in conjuntos.items():
        keep = X[idx_tr].std(0) > 0
        sx, sy = StandardScaler().fit(X[idx_tr][:, keep]), StandardScaler().fit(Y[idx_tr])
        Xtr, Xte = sx.transform(X[idx_tr][:, keep]), sx.transform(X[idx_te][:, keep])
        for nome, mdl in fl.novos_modelos(rapido=True).items():
            t0 = time.time()
            mdl.fit(Xtr, sy.transform(Y[idx_tr]))
            out[f'{nome} [{cj}]'] = fl.mape(Yte, sy.inverse_transform(mdl.predict(Xte)))
            print(f'  α={alpha}: {nome} [{cj}] {out[f"{nome} [{cj}]"].mean():.4f} %  ({time.time()-t0:.0f} s)', flush=True)
            del mdl; gc.collect()
    return pd.DataFrame(out, index=g.PARAMS)

if CACHE_FIS.exists() and not RECALCULAR:
    RES_FIS = pickle.load(open(CACHE_FIS, 'rb'))
else:
    RES_FIS = {alpha: rodar_fisica(alpha) for alpha in (0.0, ALPHA_NOM)}
    pickle.dump(RES_FIS, open(CACHE_FIS, 'wb'))

MAPE médio (%) com 5 snapshots. A coluna *entradas originais* repete a seção 3 para comparação.

In [9]:
LIN = ['Média do treino (sem estimação)', 'θ por MQ (5 snapshots)', 'θ por Bayes (5 snapshots)'] + \
      [f'{m} [{c}]' for c in ('entradas originais', 'física', 'original + física') for m in ('Ridge', 'MLP', 'LightGBM')]
def pega(t, k):
    return t.get(k.replace(' [entradas originais]', ''), np.nan)
cols = {}
for alpha, rot in ((0.0, 'sem ruído'), (ALPHA_NOM, 'com ruído')):
    t = pd.concat([RES_FIS[alpha], RES[(f'{K} snapshots', alpha)]], axis=1).mean()
    t = t[~t.index.duplicated()]
    cols[rot] = pd.Series({k: pega(t, k) for k in LIN})
T5 = pd.DataFrame(cols)
T5.round(4)

,sem ruído,com ruído
Média do treino (sem estimação),0.7979,0.7979
θ por MQ (5 snapshots),0.0000,33.3009
θ por Bayes (5 snapshots),0.0000,0.8336
Ridge [entradas originais],0.5337,0.6747
MLP [entradas originais],0.5638,0.7147
LightGBM [entradas originais],0.7082,0.7387
Ridge [física],0.0000,0.6980
MLP [física],0.0087,0.7112
LightGBM [física],0.0098,0.7087
Ridge [original + física],0.0000,0.6781


In [10]:
# razão em relação à média do treino (abaixo de 1 = a estimação ajuda)
(T5 / T5.loc[LIN[0]]).round(3)

,sem ruído,com ruído
Média do treino (sem estimação),1.000,1.000
θ por MQ (5 snapshots),0.000,41.733
θ por Bayes (5 snapshots),0.000,1.045
Ridge [entradas originais],0.669,0.846
MLP [entradas originais],0.707,0.896
LightGBM [entradas originais],0.888,0.926
Ridge [física],0.000,0.875
MLP [física],0.011,0.891
LightGBM [física],0.012,0.888
Ridge [original + física],0.000,0.850
